# Lesson 22: Projective Geometry

Lesson 8 built a hierarchy of 2D transforms &mdash; Euclidean, similarity, affine &mdash; and noted that all three preserve parallel lines. This lesson covers the next, most general step: the **projective transform (homography)**, which models what a camera actually does when it looks at a flat surface from an angle, and which does *not* preserve parallelism. That's not a bug &mdash; it's exactly the phenomenon of a vanishing point, and it's the tool behind perspective correction and image stitching.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Homogeneous coordinates

Represent a 2D point $(x, y)$ as a 3-vector $(x, y, 1)$. Any $3\times3$ matrix $H$ can then act on it by ordinary matrix multiplication; converting back to 2D means dividing by the third coordinate:

$$\begin{bmatrix}x'\\y'\\w'\end{bmatrix} = H\begin{bmatrix}x\\y\\1\end{bmatrix}, \qquad (x_{\text{2D}}', y_{\text{2D}}') = \left(\frac{x'}{w'}, \frac{y'}{w'}\right)$$

Two big payoffs: first, a $3\times3$ matrix can represent translation too (impossible with a $2\times2$ matrix alone, which is why Lesson 8 needed a separate translation vector $t$). Second, $H$ and $cH$ (any nonzero scalar multiple) represent the *exact same* transform, since the division cancels the scale &mdash; a homography has only 8 independent degrees of freedom, not 9.

## Parallel lines stop being parallel

The updated hierarchy, extending Lesson 8's table:

| Transform | Matrix form | Preserves |
|---|---|---|
| Affine | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\0&0&1\end{bmatrix}$ | parallelism |
| **Projective** | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\g&h&1\end{bmatrix}$ | straight lines only |

The only difference is a nonzero bottom row $(g, h)$ &mdash; and that alone is enough to destroy parallelism. We demonstrate directly: take two parallel horizontal lines and apply a homography with a nonzero bottom row.

In [ ]:
H = np.array([
    [1,      0.2,   0],
    [0.1,    1,     0],
    [0.0015, 0.001, 1],
], dtype=np.float64)

def apply_homography(points, H):
    homogeneous = np.hstack([points, np.ones((len(points), 1))])
    transformed = (H @ homogeneous.T).T
    return transformed[:, :2] / transformed[:, 2:3]

line1 = np.array([[0, 0], [1, 0]], dtype=np.float64)   # y = 0
line2 = np.array([[0, 1], [1, 1]], dtype=np.float64)   # y = 1, parallel to line1

l1_transformed = apply_homography(line1, H)
l2_transformed = apply_homography(line2, H)

print('line 1, transformed:', l1_transformed)
print('line 2, transformed:', l2_transformed)

extend = 400  # extend the segments far enough to visualize where they'd meet
def extended(p1, p2, length=extend):
    direction = (p2 - p1)
    return p1 - length * direction, p2 + length * direction

fig, ax = plt.subplots(figsize=(5, 4))
for line, color in [(l1_transformed, 'tab:blue'), (l2_transformed, 'tab:orange')]:
    a, b = extended(line[0], line[1])
    ax.plot([a[0], b[0]], [a[1], b[1]], color=color)
    ax.plot(line[:, 0], line[:, 1], 'o', color=color)
ax.set_xlim(-5, 20)
ax.set_ylim(-5, 5)
ax.set_title('Two originally-parallel lines, after a homography')
plt.show()

## Vanishing points, computed two ways

Where do these two lines actually meet? We can find it two ways: intersecting the transformed line segments directly, or &mdash; more elegantly &mdash; transforming the *point at infinity* in the original lines' shared direction $(1, 0)$, represented in homogeneous coordinates as $(1, 0, 0)$ (a nonzero third coordinate would make it a finite point; zero means "infinitely far away"). Applying $H$ to that point at infinity should land exactly on the vanishing point.

In [ ]:
def line_intersection(p1, p2, p3, p4):
    A = np.array([[p2[0] - p1[0], -(p4[0] - p3[0])],
                  [p2[1] - p1[1], -(p4[1] - p3[1])]])
    b = np.array([p3[0] - p1[0], p3[1] - p1[1]])
    t = np.linalg.solve(A, b)[0]
    return p1 + t * (p2 - p1)

vanishing_point_direct = line_intersection(l1_transformed[0], l1_transformed[1],
                                            l2_transformed[0], l2_transformed[1])

point_at_infinity = np.array([1.0, 0.0, 0.0])  # direction (1,0), infinitely far away
transformed_infinity = H @ point_at_infinity
vanishing_point_via_infinity = transformed_infinity[:2] / transformed_infinity[2]

print('vanishing point (line intersection): ', vanishing_point_direct)
print('vanishing point (H @ infinity trick):', vanishing_point_via_infinity)

Both methods agree exactly. This is *the* mathematical explanation for vanishing points in photographs: a homography's nonzero bottom row maps the "point at infinity" in a given direction to a genuine, finite image point, and every line in that direction converges there.

## Perspective correction: rectifying a photographed document

The most common practical use of a homography: given 4 point correspondences (e.g. the 4 corners of a document, clicked by a user or found automatically), `cv2.getPerspectiveTransform` solves for the unique homography mapping one set of 4 points to the other, and `cv2.warpPerspective` applies it.

In [ ]:
document = np.full((300, 220, 3), 255, dtype=np.uint8)
cv2.rectangle(document, (20, 20), (200, 280), (0, 0, 0), 3)
cv2.putText(document, 'HELLO', (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)

h, w = document.shape[:2]
corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
photographed_corners = np.float32([[40, 10], [w - 10, 40], [w - 30, h - 5], [10, h - 30]])  # simulated tilt

H_distort = cv2.getPerspectiveTransform(corners, photographed_corners)
photographed = cv2.warpPerspective(document, H_distort, (w, h))

H_rectify = cv2.getPerspectiveTransform(photographed_corners, corners)
rectified = cv2.warpPerspective(photographed, H_rectify, (w, h))

fig, axes = plt.subplots(1, 3, figsize=(9, 4))
for ax, im, title in zip(axes, [document, photographed, rectified],
                          ['Original document', 'Photographed at an angle\n(simulated)', 'Rectified\n(from 4 corner clicks)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

error = np.abs(rectified.astype(int) - document.astype(int))
print(f'mean abs pixel difference after distort-then-rectify round trip: {error.mean():.2f}  (small resampling loss only)')

## Solving for a homography from many correspondences

4 correspondences exactly determine a homography's 8 degrees of freedom, with no slack for error. With *more* than 4 (typically from automatic feature matching, Lesson 19), the problem becomes an overdetermined least-squares fit &mdash; the Direct Linear Transform (DLT) algorithm that `cv2.findHomography` implements, optionally wrapped in RANSAC to reject bad correspondences (mismatched features) as outliers.

In [ ]:
rng = np.random.default_rng(0)
H_true = np.array([[1, 0.2, 10], [0.05, 1, 5], [0.0008, 0.0003, 1]])

pts1 = rng.uniform(0, 200, (30, 2))
pts2 = apply_homography(pts1, H_true)

H_estimated, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)

print('true homography (scale-normalized):\n', np.round(H_true / H_true[2, 2], 4))
print('estimated homography:\n', np.round(H_estimated, 4))
print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')

With clean correspondences, `findHomography` recovers `H_true` almost exactly &mdash; this is exactly the last step of a typical image-stitching pipeline: detect and match SIFT features between two overlapping photos (Lesson 19), then solve for the homography that aligns one onto the other.

### Exercise

1. Add zero-mean Gaussian noise (e.g. std 2 pixels) to `pts2` before calling `cv2.findHomography`. How much does the estimated homography drift from `H_true`, and does increasing the number of correspondences (say, from 30 to 200) reduce that drift?
2. Deliberately corrupt a few rows of `pts2` with completely wrong values (simulating bad feature matches) and compare `cv2.findHomography(..., cv2.RANSAC, ...)` against passing `method=0` (plain least squares, no outlier rejection). How badly does the non-robust version get pulled off by the outliers?
3. In the vanishing-point demo, change the two lines to be *vertical* instead of horizontal (e.g. `x=0` and `x=1`), and predict, then verify, where their vanishing point ends up using the point-at-infinity trick with direction `(0, 1, 0)`.